# SSI FastConnect — kiểm tra REST + luồng tool của TradingAgents

Notebook này kiểm tra theo thứ tự: (1) kết nối SSI chỉ đọc, (2) lựa chọn nguồn dữ liệu trong router, (3) tool giá OHLCV và tool RSI mà các agent thực sự gọi. Không gọi LLM, không đặt lệnh.

**Credentials:** đọc từ biến môi trường runtime trước, rồi Colab Secrets; chỉ hỏi nhập ẩn nếu chưa có. Chúng chỉ ở bộ nhớ runtime, không được in ra hoặc lưu vào notebook/GitHub. Muốn xóa khỏi phiên, chọn Runtime → Disconnect and delete runtime.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/PhamHuuQan/TradingAgents.git"
BRANCH = "feature/vietnam-market-foundation"
REPO_DIR = Path("/content/TradingAgents")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

# Install the project dependencies so this checks the actual agent tools/router,
# not only the standalone SSI connector.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], cwd=str(REPO_DIR), check=True)
sys.path.insert(0, str(REPO_DIR))
print("TradingAgents branch checked out and project dependencies installed.")

In [ ]:
from datetime import datetime, timedelta
from getpass import getpass
from zoneinfo import ZoneInfo

def read_colab_secret(name):
    # Reuse the current runtime first, then Colab Secrets.
    value = os.environ.get(name)
    if value and value.strip():
        return value.strip()
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value.strip() if isinstance(value, str) and value.strip() else None

api_key = read_colab_secret("SSI_API_KEY") or getpass("SSI_API_KEY (input hidden): ")
api_secret = read_colab_secret("SSI_API_SECRET") or getpass("SSI_API_SECRET (input hidden): ")
if not api_key.strip() or not api_secret.strip():
    raise ValueError("Both SSI credentials are required; no credentials were saved to the notebook.")

os.environ["SSI_API_KEY"] = api_key.strip()
os.environ["SSI_API_SECRET"] = api_secret.strip()
os.environ["TRADINGAGENTS_CORE_STOCK_APIS"] = "ssi"
os.environ["TRADINGAGENTS_TECHNICAL_INDICATORS"] = "ssi"
del api_key, api_secret

today_vn = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).date()
from_date = (today_vn - timedelta(days=30)).isoformat()
to_date = today_vn.isoformat()

try:
    print(f"[1/3] Direct SSI REST smoke test: FPT, {from_date} to {to_date}.")
    from tradingagents.dataflows.vietnam_market.smoke import main as smoke_main
    exit_code = smoke_main(["--symbol", "FPT", "--from-date", from_date, "--to-date", to_date, "--page-size", "100"])
    if exit_code != 0:
        raise RuntimeError("SSI REST smoke test failed. Read the sanitized error above; do not share credentials.")

    print("\n[2/3] Verify that TradingAgents selected SSI for both prices and technical indicators.")
    from tradingagents.dataflows.config import set_config
    from tradingagents.dataflows.router import get_vendor
    # Make the routing assertion deterministic even if modules were imported earlier.
    set_config({"data_vendors": {"core_stock_apis": "ssi", "technical_indicators": "ssi"}})
    selected_prices = get_vendor("core_stock_apis", "get_stock_data")
    selected_indicators = get_vendor("technical_indicators", "get_indicators")
    print(f"Price-data vendor: {selected_prices}")
    print(f"Indicator vendor: {selected_indicators}")
    if selected_prices.split(",")[0].strip() != "ssi" or selected_indicators.split(",")[0].strip() != "ssi":
        raise RuntimeError("TradingAgents did not select SSI as the primary vendor; refusing to claim integration success.")

    print("\n[3/3] Call the actual agent tools (no LLM/API bill; read-only market data only).")
    from tradingagents.agents.tools import get_stock_data, get_indicators
    price_result = get_stock_data.func("FPT", from_date, to_date, to_date)
    if not isinstance(price_result, str) or "# SSI FastConnect V3 read-only OHLCV for FPT" not in price_result:
        raise RuntimeError("Agent get_stock_data tool did not return an SSI-sourced response.")
    price_rows = [line for line in price_result.splitlines() if line.strip() and not line.startswith("#") and not line.startswith("Date,")]
    if not price_rows:
        raise RuntimeError("Agent get_stock_data returned no OHLCV rows.")
    print(f"PASS — agent get_stock_data uses SSI and returned {len(price_rows)} OHLCV rows.")

    indicator_result = get_indicators.func("FPT", "rsi", to_date, 30, to_date)
    if not isinstance(indicator_result, str) or "# SSI FastConnect V3 indicator rsi for FPT" not in indicator_result:
        raise RuntimeError("Agent get_indicators tool did not return an SSI-sourced indicator.")
    indicator_rows = [line for line in indicator_result.splitlines() if line.strip() and not line.startswith("#") and not line.startswith("Date,")]
    if not indicator_rows:
        raise RuntimeError("Agent get_indicators returned no RSI values.")
    print(f"PASS — agent get_indicators uses SSI and returned {len(indicator_rows)} RSI points.")
    print("\n✅ COMPLETE: SSI REST, data routing, and both agent tools passed read-only checks.")
# Keep credentials in this private runtime's memory to avoid repeated prompts.
# They are not printed or written to a file; disconnect/delete the runtime to clear them.


## Cách đọc kết quả

- `SSI read-only connection: OK`: xác thực, REST request và chuẩn hóa dữ liệu đã chạy được.
- `Price-data vendor: ssi` và `Indicator vendor: ssi`: bộ định tuyến chọn đúng SSI.
- Hai dòng `PASS — agent ...`: các tool OHLCV và RSI mà agent sử dụng đã gọi ra dữ liệu SSI thật, không phải dữ liệu mô phỏng.
- Nếu lỗi HTTP 400/401/403, chỉ chia sẻ status và thông báo đã lọc; không gửi API Key, Secret hay access token.

Notebook không gọi LLM nên không phát sinh chi phí mô hình. Nó chỉ đọc dữ liệu thị trường, không đặt lệnh. SSI/Vnstock trong nhánh này là nguồn dùng cho phân tích hiện tại; dữ liệu lịch sử tải hôm nay không tự động là dữ liệu point-in-time hợp lệ cho backtest.